In [13]:
import glob
import os,re
import numpy as np
from typing import Dict, List, Tuple
from pathlib import Path

In [14]:
results_dir = './run_01_SlepSlep'
MASS_DIR_RE = re.compile(r"^SlepSlep_(\d+)_(\d+)_SLHA$")
SR_LINE_RE = re.compile(r"^(SR-[A-Z]{2}-[01]J[a-i])\s+\S+\s+\S+\s+(\S+)")


In [15]:
def parse_signal_file(path: Path) -> Dict[str, float]:
    """Return SR -> Acc values as strings from one signal .dat file."""
    sr_to_acc: Dict[str, float] = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        m = SR_LINE_RE.match(line.strip())
        if m:
            sr_to_acc[m.group(1)] = float(m.group(2))
        elif 'xsect' in line.strip().lower():
            xsec,unit = line.split(':')[1].split()
            if unit.strip() == 'pb':
                xsec = float(xsec) * 1e3
            else:
                xsec = float(xsec)
            sr_to_acc['xsec_fb'] = xsec
        elif 'mcevents' in line.strip().lower():
            sr_to_acc['mcevents'] = float(line.split(':')[1].strip())
    return sr_to_acc

In [16]:

results = []
for f in glob.glob(os.path.join(results_dir, "SlepSlep_*_SLHA")):
    m1,m2 = map(float, MASS_DIR_RE.match(os.path.basename(f)).groups())
    eff_dir = os.path.join(f,'analysis')
    eff_files = list(glob.glob(os.path.join(eff_dir, "*signal.dat")))
    if len(eff_files) != 1:
        print(f"Expected exactly one efficiency file in {eff_dir}, found {len(eff_files)}")
        continue
    res = {'m_slep_GeV' : m1, 'm_neut_GeV' : m2}
    res.update(parse_signal_file(Path(eff_files[0])))
    results.append(res)

In [21]:
import csv

output_csv = "SlepSlep_efficiencies.csv"

if not results:
    print("No results to export.")
else:
    # Keep key physics columns first, then append remaining SR columns in sorted order.
    preferred_cols = ["m_slep_GeV", "m_neut_GeV", "xsec_fb", "mcevents"]
    all_cols = {k for row in results for k in row.keys()}
    remaining_cols = sorted(all_cols - set(preferred_cols))
    fieldnames = [c for c in preferred_cols if c in all_cols] + remaining_cols

    sorted_results = sorted(results, key=lambda r: (r.get("m_slep_GeV", float("inf")), r.get("m_neut_GeV", float("inf"))))

    formatted_results = []
    for row in sorted_results:
        out = dict(row)
        for key, value in row.items():
            if key == "xsec_fb" or key.startswith("SR-"):
                out[key] = f"{float(value):.3e}"
        formatted_results.append(out)

    with open(output_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(formatted_results)

    print(f"Saved {len(formatted_results)} rows to {output_csv}")

Saved 84 rows to SlepSlep_efficiencies.csv
